<a href="https://colab.research.google.com/github/Pascingo/CTA-media-analysis/blob/main/Stage_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
!pip install -q -r https://raw.githubusercontent.com/Pascingo/CTA-media-analysis/main/requirements.txt

import os
import re
import json
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import gdown
from tqdm.auto import tqdm
import hashlib

# Same settings as in Stage 1
TEXT_COLUMN = "content"
START_DATE = "2024-09-01"
END_DATE = "2026-08-31"

GOVERNANCE_KEYWORDS = [
    "security", "accountability", "bias", "fairness",
    "regulatory", "regulation", "governance", "policy",
    "ethics", "compliance", "law"
]
MIN_KEYWORD_HITS = 5
RANDOM_STATE = 42

# Folders and files
DATA_FOLDER = "data"

# Stage 3 results are saved on Google Drive, so that caches survive a runtime reset
from google.colab import drive
drive.mount("/content/drive")
STAGE3_FOLDER = "/content/drive/MyDrive/CTA-media-analysis/stage3"
os.makedirs(STAGE3_FOLDER, exist_ok=True)

DRIVE_FOLDER_URL = "https://drive.google.com/drive/folders/14cAxvtxuTfwENtSDCmGcRpl6qhl-z62z"

DATASET_FILE     = os.path.join(DATA_FOLDER, "ai_media_dataset_20260911.csv")
STAGE1_GOV_FILE  = os.path.join(DATA_FOLDER, "ai_governance_cleaned.csv")
ENTITIES_FILE    = os.path.join(DATA_FOLDER, "entities_spacy.csv")
LANGEXTRACT_FILE = os.path.join(DATA_FOLDER, "langextract_ollama_25pm.jsonl")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### Download of the Stage 1 data (Pascal)

We download the files from our public Google Drive folder: the fixed dataset version (2026-09-11), the Stage 1 governance corpus, the spaCy entities and the LangExtract results. The download is skipped if the files are already in the folder `data`.

In [19]:
STAGE1_FILES = [DATASET_FILE, STAGE1_GOV_FILE, ENTITIES_FILE, LANGEXTRACT_FILE]

if not all(os.path.exists(f) for f in STAGE1_FILES):
    gdown.download_folder(DRIVE_FOLDER_URL, output=DATA_FOLDER, quiet=True)

for f in STAGE1_FILES:
    if not os.path.exists(f):
        raise FileNotFoundError(f"{f} not found. Please check the Google Drive folder.")
    print(f"{f:<45} {os.path.getsize(f) / 1e6:>8.1f} MB")

data/ai_media_dataset_20260911.csv               529.7 MB
data/ai_governance_cleaned.csv                   438.6 MB
data/entities_spacy.csv                          220.6 MB
data/langextract_ollama_25pm.jsonl                 2.7 MB


### Reconstructing the Stage 1 corpus with document IDs (Pascal)

In Stage 1 we saved the governance corpus without its index. The spaCy entities and the LangExtract results, however, refer to the articles by this index (`doc_id`). We therefore repeat the cleaning steps of Stage 1 on the original dataset. The steps are deterministic, so we get the same articles with the same `doc_id`.

Two checks make sure this worked:

1. The number of articles and the titles must be identical to `ai_governance_cleaned.csv`.
2. All `doc_id`s in the LangExtract results must exist in the reconstructed corpus.

We keep the **full cleaned corpus** (`df`), and not only the 12,393 Stage 1 articles. The new topic filter in Part 1 has to be evaluated on all articles, so that we also see relevant articles that the Stage 1 filter missed (recall). The column `in_stage1` marks the articles of the Stage 1 corpus.

In [ ]:
# 1. Same cleaning as in Stage 1 (sections 2 and 3)
df = pd.read_csv(DATASET_FILE, usecols=["title", "date", "content", "domain", "url", "tags"])
df = df.dropna(subset=[TEXT_COLUMN])
df = df.drop_duplicates(subset=[TEXT_COLUMN])

df["published_at"] = pd.to_datetime(df["date"], errors="coerce")
df = df[(df["published_at"] >= START_DATE) & (df["published_at"] <= END_DATE)].copy()

df["cleaned_text"] = df[TEXT_COLUMN].apply(
    lambda t: unicodedata.normalize("NFKD", t).encode("ascii", "ignore").decode()
)
df["cleaned_text"] = (
    df["cleaned_text"]
    .str.lower()
    .str.replace(r"<[^>]+>", " ", regex=True)
    .str.replace(r"[-/]", " ", regex=True)
    .str.replace(r"[^a-z0-9\s.,!?]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

pattern = r"\b(?:" + "|".join(GOVERNANCE_KEYWORDS) + r")\b"
df["gov_keyword_hits"] = df["cleaned_text"].str.count(pattern)
df["in_stage1"] = df["gov_keyword_hits"] >= MIN_KEYWORD_HITS
df.index.name = "doc_id"

# 2. Check 1: same articles as the saved Stage 1 corpus
df_gov_s1 = pd.read_csv(STAGE1_GOV_FILE)
stage1_ids = df.index[df["in_stage1"]]

assert len(stage1_ids) == len(df_gov_s1), "Number of articles differs from Stage 1."
assert (
    df.loc[stage1_ids, "title"].fillna("").values == df_gov_s1["title"].fillna("").values
).all(), "Titles differ from Stage 1."

# Take over the columns computed in Stage 1 (spaCy lemmas, sentiment)
for col in ["processed_text", "sentiment_polarity"]:
    df.loc[stage1_ids, col] = df_gov_s1[col].values

# 3. Check 2: the LangExtract results refer to existing articles
lx_doc_ids = pd.read_json(LANGEXTRACT_FILE, lines=True)["doc_id"]
share_found = lx_doc_ids.isin(stage1_ids).mean()
assert share_found == 1.0, f"Only {share_found:.1%} of the LangExtract doc_ids were found."

print(f"Full cleaned corpus:        {len(df):>6,} articles")
print(f"Stage 1 governance corpus:  {df['in_stage1'].sum():>6,} articles")
print(f"LangExtract articles found: {lx_doc_ids.nunique():>6,} (100%)")

# 1. Addressing the Stage 1 feedback

The feedback on Stage 1 named three problems:

1. Several topics of our corpus are about **broader politics, energy and enterprise technology** rather than AI Governance. Our topic filter was too broad.
2. The relation extraction produced many **inconsistent predicates** and **source-alignment problems**. After schema filtering, very few factual relations remained.
3. The co-occurrence graph is useful for exploration, but it is **not factual evidence**.

These problems directly affect the RAG system: irrelevant articles become irrelevant retrieved passages, and wrong relations become wrong "facts" in the answers. We therefore address them first.

## 1.1 Tightened topic filter (Pascal)

The Stage 1 filter counted eleven general keywords and kept every article with at least 5 hits. It had two weaknesses:

- **Generic words** such as *security*, *policy* and *law* also appear in articles about cybersecurity products, US domestic politics or company policies.
- **Absolute counts** favour long articles: a 4,000-word article about energy markets reaches 5 hits easily, even if governance is only a side remark.

The new filter works in three steps:

1. **Governance score:** We count specific governance terms in two groups. *Strong terms* (e.g. *AI Act*, *code of practice*, *AI safety institute*, *copyright*, *lawsuit*) are clearly about governance on their own and get weight 3. *Medium terms* (e.g. *regulation*, *accountability*, *bias*, *oversight*) get weight 1. The generic words *security*, *policy* and *law* are no longer counted. We divide the weighted hits by the article length (**hits per 1,000 words**), so that long articles are not favoured.
2. **Off-topic score:** For the three confounders named in the feedback (politics, energy, enterprise technology), we count typical terms in the same way. The term lists are based on our Stage 1 topics 2, 3, 4, 5 and 7.
3. **Decision rule:** An article is kept if governance is dense enough, appears in the title or across several distinct terms, and is **at least as dense as the strongest off-topic vocabulary**. In other words, governance has to be the main subject, not a side remark.

The thresholds in this section are **provisional**. In section 1.2 we evaluate the filter on a manually annotated sample and choose the final thresholds there.

In [ ]:
import hashlib

# Strong terms: clearly about AI Governance on their own (weight 3)
STRONG_TERMS = [
    r"ai act", r"code of practice", r"ai (?:safety|security) institutes?",
    r"executive orders?", r"gdpr", r"digital services act", r"responsible ai",
    r"ai governance", r"ai regulations?", r"ai safety", r"ai ethics",
    r"algorithmic bias", r"sb 53", r"sb 1047", r"frontier ai", r"deepfakes?",
    r"copyright", r"lawsuits?", r"data protection", r"ai liability",
    # AI security is part of our track
    r"ai security", r"prompt injections?", r"adversarial attacks?", r"(?:data|model) poisoning",
]

# Medium terms: typical governance vocabulary (weight 1)
MEDIUM_TERMS = [
    r"regulations?", r"regulatory", r"regulators?", r"legislation", r"lawmakers?",
    r"governance", r"accountability", r"fairness", r"bias(?:es|ed)?", r"discrimination",
    r"ethic(?:s|al)", r"transparency", r"oversight", r"liability", r"compliance",
    r"privacy", r"audits?", r"watermark(?:s|ing)?", r"red teaming", r"jailbreaks?",
    r"misinformation", r"ai risks?",
]

# Off-topic vocabulary for the three confounders named in the feedback
OFFTOPIC_TERMS = {
    "politics": [
        r"elections?", r"campaigns?", r"voters?", r"democrats?", r"republicans?",
        r"tariffs?", r"trade war", r"immigration", r"ukraine", r"russia",
        r"israel", r"gaza", r"iran", r"nato", r"military",
    ],
    "energy": [
        r"energy", r"electricity", r"power grids?", r"grid", r"nuclear",
        r"gigawatts?", r"megawatts?", r"utilities", r"oil", r"gas", r"solar",
        r"emissions", r"data centers?",
    ],
    "enterprise": [
        r"revenue", r"earnings", r"quarterly", r"shares", r"stock", r"investors?",
        r"valuation", r"funding round", r"ipo", r"customers?", r"enterprises?",
        r"saas", r"hybrid cloud", r"red hat", r"kubernetes", r"apis?",
        r"developers?", r"workflows?", r"productivity", r"mcp",
    ],
}


def count_terms(texts, terms, desc):
    """Count every term in every text in ONE pass per text.
    Each term is its own capture group; match.lastindex tells us which term matched."""
    regex = re.compile("|".join(rf"(\b{t}\b)" for t in terms))
    counts = np.zeros((len(texts), len(terms)), dtype=np.int32)
    for i, text in enumerate(tqdm(texts, desc=desc)):
        for match in regex.finditer(text):
            counts[i, match.lastindex - 1] += 1
    return pd.DataFrame(counts, columns=terms, index=texts.index)


# Cache: the file name contains a hash of the term lists,
# so the features are recomputed automatically when a list changes
term_hash = hashlib.md5(
    json.dumps([STRONG_TERMS, MEDIUM_TERMS, OFFTOPIC_TERMS]).encode()
).hexdigest()[:8]
TOPIC_FEATURES_FILE = os.path.join(STAGE3_FOLDER, f"topic_features_{term_hash}.parquet")

df["cleaned_title"] = (
    df["title"].fillna("").astype(str).str.lower()
    .str.replace(r"[-/]", " ", regex=True)
    .str.replace(r"[^a-z0-9\s.,!?]", "", regex=True)
)
df["n_words"] = df["cleaned_text"].str.count(" ") + 1

if os.path.exists(TOPIC_FEATURES_FILE):
    features = pd.read_parquet(TOPIC_FEATURES_FILE)
    print("Loaded topic features from cache.")
else:
    strong_counts = count_terms(df["cleaned_text"], STRONG_TERMS, "strong terms")
    medium_counts = count_terms(df["cleaned_text"], MEDIUM_TERMS, "medium terms")

    features = pd.DataFrame(index=df.index)
    features["gov_strong_hits"] = strong_counts.sum(axis=1)
    features["gov_medium_hits"] = medium_counts.sum(axis=1)
    features["gov_distinct_terms"] = (strong_counts > 0).sum(axis=1) + (medium_counts > 0).sum(axis=1)

    title_pattern = "|".join(rf"\b{t}\b" for t in STRONG_TERMS + MEDIUM_TERMS)
    features["gov_in_title"] = df["cleaned_title"].str.contains(title_pattern, regex=True)

    for category, terms in OFFTOPIC_TERMS.items():
        features[f"off_{category}_hits"] = count_terms(df["cleaned_text"], terms, category).sum(axis=1)

    features.to_parquet(TOPIC_FEATURES_FILE)

df = df.drop(columns=features.columns, errors="ignore").join(features)

# Densities per 1,000 words
df["gov_score_per_1k"] = 1000 * (3 * df["gov_strong_hits"] + df["gov_medium_hits"]) / df["n_words"]
for category in OFFTOPIC_TERMS:
    df[f"off_{category}_per_1k"] = 1000 * df[f"off_{category}_hits"] / df["n_words"]

OFF_COLUMNS = [f"off_{c}_per_1k" for c in OFFTOPIC_TERMS]
df["off_max_per_1k"] = df[OFF_COLUMNS].max(axis=1)
df["off_main_category"] = df[OFF_COLUMNS].idxmax(axis=1).str.replace("off_", "").str.replace("_per_1k", "")

df[["n_words", "gov_score_per_1k", "gov_distinct_terms"] + OFF_COLUMNS].describe().round(2)

### Lexical candidate filter (Pascal)

A first run showed that the lexical score alone is not precise enough. Articles about healthcare, construction, solar panels or tax reached a governance score of 5–9 only through general words such as *privacy*, *compliance* or *transparency*. Such topics cannot all be covered by off-topic lists. The lexical step is therefore used as a **high-recall candidate filter**: it should keep almost all real governance articles, and precision is added in the next step with a semantic score.

An article becomes a candidate (`gov_lexical`) if all three conditions are met:

1. `gov_score_per_1k >= MIN_GOV_SCORE`;
2. the title contains a governance term, **or** the text contains at least `MIN_DISTINCT_TERMS` different governance terms;
3. `gov_score_per_1k >= off_max_per_1k`: governance is at least as dense as the strongest off-topic vocabulary.

We also added four AI-security terms (e.g. *prompt injection*, *data poisoning*). The first run had removed an article about securing agentic AI systems, although AI security is part of our track.

In [ ]:
# Loose thresholds: the candidate filter should favour recall
MIN_GOV_SCORE = 3
MIN_DISTINCT_TERMS = 3


def apply_lexical_filter(data, min_score, min_distinct):
    return (
        (data["gov_score_per_1k"] >= min_score)
        & (data["gov_in_title"] | (data["gov_distinct_terms"] >= min_distinct))
        & (data["gov_score_per_1k"] >= data["off_max_per_1k"])
    )


for min_score in [2, 3, 5, 8]:
    n = apply_lexical_filter(df, min_score, MIN_DISTINCT_TERMS).sum()
    print(f"gov_score_per_1k >= {min_score}: {n:>6,} articles ({n / len(df):.1%})")

df["gov_lexical"] = apply_lexical_filter(df, MIN_GOV_SCORE, MIN_DISTINCT_TERMS)

print()
display(pd.crosstab(df["in_stage1"], df["gov_lexical"], margins=True))

### First check: which articles change? (Pascal)

We compare three groups: the articles that **both** filters keep, the articles that **only Stage 1** kept (removed by the new filter), and the articles that **only the new filter** keeps (added). If the new filter works as intended, the removed articles should have a much higher off-topic density. A random sample of titles from each group gives a first qualitative impression.

In [ ]:
# Remove the column of the first, discarded filter version
df = df.drop(columns=["gov_v2"], errors="ignore")

groups = {
    "kept by both":           df["in_stage1"] & df["gov_lexical"],
    "removed (only Stage 1)": df["in_stage1"] & ~df["gov_lexical"],
    "added (only lexical)":   ~df["in_stage1"] & df["gov_lexical"],
}

comparison = pd.DataFrame({
    name: df.loc[mask, ["gov_score_per_1k"] + OFF_COLUMNS].mean() for name, mask in groups.items()
}).T
comparison.insert(0, "articles", [mask.sum() for mask in groups.values()])
display(comparison.round(2))

print("Main off-topic category of the removed articles:")
display(df.loc[groups["removed (only Stage 1)"], "off_main_category"].value_counts())

for name, mask in groups.items():
    print(f"\n--- {name} ---")
    sample = df.loc[mask].sample(n=min(10, mask.sum()), random_state=RANDOM_STATE)
    for title, score, off in zip(sample["title"], sample["gov_score_per_1k"], sample["off_max_per_1k"]):
        print(f"gov={score:5.1f}  off={off:5.1f}  {str(title)[:110]}")